# LSTM Training
Composite loss: `money_pct + 0.2 * smape`. Recursive 48h inference over month-long val/test.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import os
import math
import time
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from typing import Dict, List, Tuple
from tqdm.notebook import tqdm
import mlflow

from training.data_loading import *
from training.loss_funcs import *

print(f"PyTorch version  : {torch.__version__}")
print(f"CUDA available   : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU              : {torch.cuda.get_device_name(0)}")
    torch.set_float32_matmul_precision("medium")
    print("float32 matmul precision set to 'medium'")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SEED = 42
torch.manual_seed(SEED); np.random.seed(SEED)

In [ ]:
weather_cols_all = ['temperature_2m',
       'apparent_temperature', 'dew_point_2m', 'relative_humidity_2m',
       'precipitation', 'rain', 'snowfall', 'cloud_cover', 'cloud_cover_low',
       'cloud_cover_mid', 'cloud_cover_high', 'surface_pressure',
       'wind_speed_10m', 'wind_direction_10m', 'wind_gusts_10m',
       'shortwave_radiation', 'diffuse_radiation', 'direct_normal_irradiance']

other_cols = [
    'dam_price', 'buy_bm_price', 'sell_bm_price',
    'max_power', 'max_solar', 'max_ev'
]

cat_columns = [
    'eic_code', 'dso_desc', 'station_type', 'oblast',
    'Month', 'Day', 'Hour', 'day_of_week', 'season'
]

static_cols = [
    'latitude', 'longitude', 'eic_code', 'dso_desc', 'station_type', 'oblast'
]

calendar_cols = [
    'Month', 'Day', 'Hour', 'day_of_week', 'season'
]

time_cols = ['datetime', 'time_idx']

FUTURE_REALS = weather_cols_all + calendar_cols + static_cols + other_cols
PRICE_COLS = ['dam_price', 'sell_bm_price', 'buy_bm_price']

print(f"y col is: {Y_COL}, group col is: {GROUP_COL}\n"
      f"# features: {len(FUTURE_REALS)}")

In [ ]:
# OHE data — LSTM consumes numeric tensors only.
print("Loading train …")
train = load_and_prepare(TRAIN_PATH_OHE)

print("Loading val   …")
val = load_and_prepare(VAL_PATH_OHE)

print("Loading test  …")
test = load_and_prepare(TEST_PATH_OHE)

print(f"train: {train.shape}")
print(f"val  : {val.shape}")
print(f"test : {test.shape}")

In [ ]:
# OHE the group col (the only cat col left in OHE data) and remember the encoded columns
# so we can re-attach datetime/prices from the raw frames later for evaluation.
print("Loading raw frames (kept aside for datetime + prices during eval) …")
train_raw = load_and_prepare(TRAIN_PATH_WITH_DATETME)
val_raw   = load_and_prepare(VAL_PATH_WITH_DATETME)
test_raw  = load_and_prepare(TEST_PATH_WITH_DATETME)

# OHE the group col across all three frames consistently
all_groups = pd.concat([train[GROUP_COL], val[GROUP_COL], test[GROUP_COL]]).unique()
group_to_idx = {g: i for i, g in enumerate(sorted(all_groups))}
n_groups = len(group_to_idx)
print(f"# groups: {n_groups}")

for df in (train, val, test):
    df['__gidx'] = df[GROUP_COL].map(group_to_idx).astype(np.int64)

In [ ]:
# Optional sub-sample for speed. Comment out to use all stations.
TARGET_STATIONS = 395

station_stats = (
    train.groupby(GROUP_COL)
    .agg(rows=(Y_COL, "count"))
    .reset_index()
    .sort_values("rows", ascending=False)
)
sampled_stations = station_stats.sample(n=TARGET_STATIONS, random_state=SEED)[GROUP_COL].values
print(f"Stations sampled: {len(sampled_stations)}")

train     = train[train[GROUP_COL].isin(sampled_stations)].reset_index(drop=True)
val       = val[val[GROUP_COL].isin(sampled_stations)].reset_index(drop=True)
test      = test[test[GROUP_COL].isin(sampled_stations)].reset_index(drop=True)
train_raw = train_raw[train_raw[GROUP_COL].isin(sampled_stations)].reset_index(drop=True)
val_raw   = val_raw[val_raw[GROUP_COL].isin(sampled_stations)].reset_index(drop=True)
test_raw  = test_raw[test_raw[GROUP_COL].isin(sampled_stations)].reset_index(drop=True)

print(f"Train rows : {len(train):,}")
print(f"Val rows   : {len(val):,}")
print(f"Test rows  : {len(test):,}")

In [ ]:
training_cutoff = train["time_idx"].max()
val_cutoff      = val["time_idx"].max()
test_cutoff     = test["time_idx"].max()

print(f"training cutoff : {training_cutoff}")
print(f"val cutoff      : {val_cutoff}")
print(f"test cutoff     : {test_cutoff}")

## Feature columns and scaling
Build the numeric feature matrix. After OHE the group col is removed from features (we kept `__gidx` for grouping/identification only — we don't feed it to the LSTM).

In [ ]:
# All numeric columns except: target, time_idx, group identifier, and any leftover identifier cols.
non_feature = {Y_COL, 'time_idx', '__gidx', GROUP_COL}
feature_cols = [c for c in train.columns if c not in non_feature and pd.api.types.is_numeric_dtype(train[c])]
print(f"# input features: {len(feature_cols)}")

# Standardize using train stats only.
feat_mean = train[feature_cols].mean().values.astype(np.float32)
feat_std  = train[feature_cols].std().replace(0, 1).values.astype(np.float32)

y_mean = float(train[Y_COL].mean())
y_std  = float(train[Y_COL].std() or 1.0)
print(f"y_mean={y_mean:.4f}  y_std={y_std:.4f}")

def standardize_features(df):
    arr = df[feature_cols].values.astype(np.float32)
    return (arr - feat_mean) / feat_std

def standardize_y(y):
    return (y - y_mean) / y_std

def destandardize_y(y):
    return y * y_std + y_mean

## Composite loss: `money_pct + 0.2 * smape`
Both terms are differentiable PyTorch ops. After defining `money_pct_torch` we run a sanity check against `money_pct` from `loss_funcs.py` on sample data.

In [ ]:
def smape_torch(y_true: torch.Tensor, y_pred: torch.Tensor, eps: float = 1e-8) -> torch.Tensor:
    num = torch.abs(y_pred - y_true)
    den = (torch.abs(y_true) + torch.abs(y_pred)) / 2.0 + eps
    return 100.0 * torch.mean(num / den)

def money_pct_torch(y_true: torch.Tensor,
                    y_pred: torch.Tensor,
                    price: torch.Tensor,
                    selling_price: torch.Tensor,
                    buying_price: torch.Tensor,
                    eps: float = 1e-8) -> torch.Tensor:
    """
    Differentiable port of money_pct.
    Imbalance settlement: actual revenue =
        y_pred * dam_price
        + max(y_true - y_pred, 0) * sell_bm_price          (over-produced -> sell surplus)
        - max(y_pred - y_true, 0) * buy_bm_price           (under-produced -> buy shortfall)
    Ideal revenue = y_true * dam_price.
    money_pct = 100 * (ideal - actual) / |ideal|.   (lower is better; 0 = perfect)
    """
    surplus  = torch.clamp(y_true - y_pred, min=0.0)
    shortage = torch.clamp(y_pred - y_true, min=0.0)
    actual = y_pred * price + surplus * selling_price - shortage * buying_price
    ideal  = y_true * price
    return 100.0 * (ideal.sum() - actual.sum()) / (torch.abs(ideal.sum()) + eps)

def composite_loss(y_true, y_pred, price, sell_p, buy_p, smape_w: float = 0.2):
    return money_pct_torch(y_true, y_pred, price, sell_p, buy_p) + smape_w * smape_torch(y_true, y_pred)

# ---- Sanity check vs reference money_pct from loss_funcs.py ----
_rng = np.random.default_rng(0)
_n = 256
_y_true  = _rng.uniform(0, 100, _n).astype(np.float32)
_y_pred  = (_y_true + _rng.normal(0, 5, _n)).astype(np.float32)
_dam     = _rng.uniform(50, 150, _n).astype(np.float32)
_sell_bm = (_dam * _rng.uniform(0.6, 0.95, _n)).astype(np.float32)
_buy_bm  = (_dam * _rng.uniform(1.05, 1.4, _n)).astype(np.float32)

_ref = money_pct(_y_true, _y_pred, _dam, _sell_bm, _buy_bm)
_ours = money_pct_torch(
    torch.from_numpy(_y_true), torch.from_numpy(_y_pred),
    torch.from_numpy(_dam), torch.from_numpy(_sell_bm), torch.from_numpy(_buy_bm),
).item()
print(f"money_pct (loss_funcs): {_ref:.6f}")
print(f"money_pct_torch      : {_ours:.6f}")
print(f"abs diff             : {abs(_ref - _ours):.2e}  -> {'MATCH' if abs(_ref - _ours) < 1e-3 else 'MISMATCH (review formula!)'}")

## Sequence dataset
Per group, build sliding windows: encoder length `L` of past observations -> forecast horizon `H` of future steps. The target is the standardized `Y_COL` over the horizon. We also carry the price columns (un-standardized) over the horizon for the loss.

In [ ]:
L = 168   # encoder length (1 week)
H = 48    # forecast horizon (2 days)

price_idx = [feature_cols.index(c) for c in PRICE_COLS]  # within standardized features
# Means/stds for de-standardizing prices for the loss (loss expects raw price scale)
price_means = feat_mean[price_idx]
price_stds  = feat_std[price_idx]

def build_group_arrays(df: pd.DataFrame, raw_df: pd.DataFrame):
    """Returns dict: gidx -> dict(X, y, prices_raw, time_idx).
    Sorted by time_idx within group.
    `raw_df` is used to fetch raw price columns aligned by (group, time_idx)."""
    out = {}
    df = df.sort_values(['__gidx', 'time_idx']).reset_index(drop=True)
    raw_df = raw_df.sort_values([GROUP_COL, 'time_idx']).reset_index(drop=True)
    raw_lookup = raw_df.set_index([GROUP_COL, 'time_idx'])[PRICE_COLS]

    X_full = standardize_features(df)
    y_full = standardize_y(df[Y_COL].values.astype(np.float32))
    t_full = df['time_idx'].values.astype(np.int64)
    g_full = df['__gidx'].values
    grp_full = df[GROUP_COL].values

    # Reconstruct raw prices in row order of df
    keys = list(zip(grp_full, t_full))
    prices_raw = raw_lookup.loc[keys].values.astype(np.float32)

    for gidx in np.unique(g_full):
        m = g_full == gidx
        out[int(gidx)] = {
            'X': X_full[m],
            'y': y_full[m],
            'prices_raw': prices_raw[m],
            'time_idx': t_full[m],
        }
    return out

print("Building per-group arrays …")
train_g = build_group_arrays(train, train_raw)
val_g   = build_group_arrays(val,   val_raw)
test_g  = build_group_arrays(test,  test_raw)
print(f"train groups: {len(train_g)}, val groups: {len(val_g)}, test groups: {len(test_g)}")

In [ ]:
class SeqDataset(Dataset):
    """Sliding-window dataset across all groups."""
    def __init__(self, group_arrays: Dict[int, Dict], L: int, H: int, stride: int = 1):
        self.L, self.H = L, H
        self.index = []  # list of (gidx, start)
        self.group_arrays = group_arrays
        for gidx, d in group_arrays.items():
            n = len(d['y'])
            if n < L + H:
                continue
            for s in range(0, n - L - H + 1, stride):
                self.index.append((gidx, s))

    def __len__(self): return len(self.index)

    def __getitem__(self, i):
        gidx, s = self.index[i]
        d = self.group_arrays[gidx]
        L, H = self.L, self.H
        x_enc   = d['X'][s : s + L]                           # (L, F)
        x_dec   = d['X'][s + L : s + L + H]                   # (H, F)  — known future covariates
        y_enc   = d['y'][s : s + L]                           # (L,)
        y_tgt   = d['y'][s + L : s + L + H]                   # (H,)    standardized
        prices  = d['prices_raw'][s + L : s + L + H]          # (H, 3)  raw
        return (
            torch.from_numpy(x_enc),
            torch.from_numpy(y_enc).unsqueeze(-1),
            torch.from_numpy(x_dec),
            torch.from_numpy(y_tgt),
            torch.from_numpy(prices),
        )

BATCH = 256
STRIDE_TRAIN = 4   # 4h stride keeps things tractable
STRIDE_EVAL  = H   # non-overlapping windows for eval metric printing

train_ds = SeqDataset(train_g, L, H, stride=STRIDE_TRAIN)
val_ds   = SeqDataset(val_g,   L, H, stride=STRIDE_EVAL)

print(f"train windows: {len(train_ds):,}")
print(f"val windows  : {len(val_ds):,}")

train_loader = DataLoader(train_ds, batch_size=BATCH, shuffle=True,  num_workers=0, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH, shuffle=False, num_workers=0, pin_memory=True)

## LSTM model
Encoder-decoder LSTM. Encoder consumes past `(features + y)`. Decoder consumes known future features and previous y prediction (teacher forcing during training, autoregressive at inference).

In [ ]:
class LSTMForecaster(nn.Module):
    def __init__(self, n_feat: int, hidden: int = 128, layers: int = 2, dropout: float = 0.2):
        super().__init__()
        self.encoder = nn.LSTM(
            input_size=n_feat + 1,  # features + y
            hidden_size=hidden, num_layers=layers,
            batch_first=True, dropout=dropout if layers > 1 else 0.0,
        )
        self.decoder = nn.LSTM(
            input_size=n_feat + 1,  # features + previous y
            hidden_size=hidden, num_layers=layers,
            batch_first=True, dropout=dropout if layers > 1 else 0.0,
        )
        self.head = nn.Linear(hidden, 1)

    def forward(self, x_enc, y_enc, x_dec, y_tgt=None, teacher_forcing: float = 1.0):
        # Encode
        enc_in = torch.cat([x_enc, y_enc], dim=-1)
        _, (h, c) = self.encoder(enc_in)

        B, H, F = x_dec.shape
        prev_y = y_enc[:, -1:, :]  # (B,1,1)
        outs = []
        for t in range(H):
            step_in = torch.cat([x_dec[:, t:t+1, :], prev_y], dim=-1)
            out, (h, c) = self.decoder(step_in, (h, c))
            yhat = self.head(out)   # (B,1,1)
            outs.append(yhat)
            if y_tgt is not None and torch.rand(1).item() < teacher_forcing:
                prev_y = y_tgt[:, t:t+1].unsqueeze(-1)
            else:
                prev_y = yhat
        return torch.cat(outs, dim=1).squeeze(-1)  # (B, H)

n_feat = len(feature_cols)
model = LSTMForecaster(n_feat=n_feat, hidden=128, layers=2, dropout=0.2).to(DEVICE)
print(model)
n_params = sum(p.numel() for p in model.parameters())
print(f"# params: {n_params:,}")

## Training
After every eval/epoch, prints metrics in the requested format:
```
train_pct=12.767% train_smape=6.687 val_pct=12.767% val_smape=6.687
```

In [ ]:
EPOCHS = 15
LR = 1e-3
WD = 1e-5
TF_START, TF_END = 1.0, 0.3   # teacher forcing decay
SMAPE_W = 0.2

opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WD)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS)

# Convert price (standardized in features) -> raw scale for the loss.
# We pass raw prices directly via the dataset; no need to undo standardization here.

def run_epoch(loader, train_mode: bool, tf: float, desc: str):
    model.train(train_mode)
    total_loss = 0.0
    pct_num = 0.0; pct_den = 0.0  # for aggregate money_pct
    smape_num = 0.0; smape_den = 0.0
    n_seen = 0

    pbar = tqdm(loader, desc=desc, leave=False)
    for x_enc, y_enc, x_dec, y_tgt, prices in pbar:
        x_enc = x_enc.to(DEVICE, non_blocking=True)
        y_enc = y_enc.to(DEVICE, non_blocking=True)
        x_dec = x_dec.to(DEVICE, non_blocking=True)
        y_tgt = y_tgt.to(DEVICE, non_blocking=True)
        prices = prices.to(DEVICE, non_blocking=True)

        with torch.set_grad_enabled(train_mode):
            y_hat_std = model(x_enc, y_enc, x_dec, y_tgt=y_tgt if train_mode else None,
                              teacher_forcing=tf if train_mode else 0.0)
            # de-standardize for loss (prices are in raw scale)
            y_hat = y_hat_std * y_std + y_mean
            y_true = y_tgt * y_std + y_mean

            dam   = prices[..., 0].reshape(-1)
            sellp = prices[..., 1].reshape(-1)
            buyp  = prices[..., 2].reshape(-1)
            yt_flat = y_true.reshape(-1)
            yp_flat = y_hat.reshape(-1)

            loss = composite_loss(yt_flat, yp_flat, dam, sellp, buyp, smape_w=SMAPE_W)

        if train_mode:
            opt.zero_grad(set_to_none=True)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()

        # Aggregate batch metrics into running totals
        bs = yt_flat.numel()
        n_seen += bs
        total_loss += loss.item() * bs

        with torch.no_grad():
            # money_pct accumulates correctly only when summed across the whole epoch
            ideal = (yt_flat * dam).sum().item()
            surplus  = torch.clamp(yt_flat - yp_flat, min=0.0)
            shortage = torch.clamp(yp_flat - yt_flat, min=0.0)
            actual = (yp_flat * dam + surplus * sellp - shortage * buyp).sum().item()
            pct_num += (ideal - actual)
            pct_den += abs(ideal)

            sm_n = (yp_flat - yt_flat).abs().sum().item()
            sm_d = ((yt_flat.abs() + yp_flat.abs()) / 2.0 + 1e-8).sum().item()
            smape_num += sm_n
            smape_den += sm_d

        pbar.set_postfix(loss=f"{loss.item():.3f}")

    avg_loss = total_loss / max(n_seen, 1)
    money_pct_epoch = 100.0 * pct_num / max(pct_den, 1e-8)
    smape_epoch     = 100.0 * smape_num / max(smape_den, 1e-8)
    return avg_loss, money_pct_epoch, smape_epoch

mlflow.set_experiment("lstm_forecast")
mlflow.start_run()
mlflow.log_params({
    "L": L, "H": H, "batch": BATCH, "epochs": EPOCHS, "lr": LR, "wd": WD,
    "hidden": 128, "layers": 2, "dropout": 0.2, "smape_w": SMAPE_W,
    "stride_train": STRIDE_TRAIN, "stride_eval": STRIDE_EVAL,
    "n_features": n_feat, "n_groups": n_groups, "n_params": n_params,
})

best_val = float("inf")
for epoch in range(1, EPOCHS + 1):
    tf = TF_START + (TF_END - TF_START) * (epoch - 1) / max(EPOCHS - 1, 1)
    t0 = time.time()

    tr_loss, tr_pct, tr_smape = run_epoch(train_loader, train_mode=True,  tf=tf, desc=f"epoch {epoch} train")
    va_loss, va_pct, va_smape = run_epoch(val_loader,   train_mode=False, tf=0.0, desc=f"epoch {epoch} val")

    print(
        f"[epoch {epoch:02d}/{EPOCHS}] "
        f"train_loss={tr_loss:.4f} val_loss={va_loss:.4f} | "
        f"train_pct={tr_pct:.3f}% train_smape={tr_smape:.3f} "
        f"val_pct={va_pct:.3f}% val_smape={va_smape:.3f} "
        f"(tf={tf:.2f}, {time.time()-t0:.1f}s)"
    )

    mlflow.log_metrics({
        "train_loss": tr_loss, "val_loss": va_loss,
        "train_pct": tr_pct,   "train_smape": tr_smape,
        "val_pct": va_pct,     "val_smape": va_smape,
        "tf": tf,
    }, step=epoch)

    sched.step()

    if va_loss < best_val:
        best_val = va_loss
        torch.save(model.state_dict(), "lstm_best.pt")
        print(f"  -> saved lstm_best.pt (val_loss={va_loss:.4f})")

model.load_state_dict(torch.load("lstm_best.pt"))
print("Loaded best model.")

## Recursive inference
Val and test span ~1 month each. The model only forecasts `H=48` ahead, so we roll the encoder window forward `H` steps at a time, feeding back our predictions into the next encoder window.

In [ ]:
@torch.no_grad()
def recursive_forecast(group_arrays: Dict[int, Dict], raw_df: pd.DataFrame) -> pd.DataFrame:
    """Forecast every step of the eval period for every group, by rolling H-step windows.
    Initial encoder context comes from the last L points of *train* — for val we'll use the
    end of the train arrays as warm-up. Here we simplify: use the first L points of each
    group as warm-up (encoder seed) and forecast the remainder. This means the first L
    points of the eval period are not predicted (they are the seed) — they are excluded
    from the eval frame."""
    model.eval()
    rows = []
    raw_df = raw_df.sort_values([GROUP_COL, 'time_idx']).reset_index(drop=True)
    raw_lookup = raw_df.set_index([GROUP_COL, 'time_idx'])

    # Reverse map gidx -> group code
    idx_to_group = {v: k for k, v in group_to_idx.items()}

    for gidx, d in tqdm(group_arrays.items(), desc="recursive forecast"):
        X = d['X']; y = d['y']; t = d['time_idx']; pr = d['prices_raw']
        n = len(y)
        if n < L + H:
            continue
        # Encoder seed = first L points (true y, standardized)
        y_run = y.copy()  # we will overwrite predicted positions with predictions
        cursor = L
        preds_std = np.full(n, np.nan, dtype=np.float32)
        while cursor < n:
            h_step = min(H, n - cursor)
            x_enc = torch.from_numpy(X[cursor - L:cursor]).unsqueeze(0).to(DEVICE)
            y_enc = torch.from_numpy(y_run[cursor - L:cursor]).unsqueeze(0).unsqueeze(-1).to(DEVICE)
            x_dec = torch.from_numpy(X[cursor:cursor + h_step]).unsqueeze(0).to(DEVICE)
            yhat = model(x_enc, y_enc, x_dec, y_tgt=None, teacher_forcing=0.0)  # (1, h_step)
            yhat_np = yhat.squeeze(0).detach().cpu().numpy()
            preds_std[cursor:cursor + h_step] = yhat_np
            y_run[cursor:cursor + h_step] = yhat_np  # feed back
            cursor += h_step

        grp_code = idx_to_group[gidx]
        # Build per-step rows for indexes >= L (the predicted part)
        for i in range(L, n):
            tt = int(t[i])
            try:
                rec = raw_lookup.loc[(grp_code, tt)]
                dt  = rec['datetime']
                ytrue = rec[Y_COL]
                dam   = rec['dam_price']; sellp = rec['sell_bm_price']; buyp = rec['buy_bm_price']
            except KeyError:
                continue
            rows.append((grp_code, tt, dt, float(ytrue), float(destandardize_y(preds_std[i])),
                         float(dam), float(sellp), float(buyp)))

    out = pd.DataFrame(rows, columns=[GROUP_COL, 'time_idx', 'datetime', Y_COL, 'pred',
                                      'dam_price', 'sell_bm_price', 'buy_bm_price'])
    return out

print("Forecasting val …")
val_eval = recursive_forecast(val_g, val_raw)
print(f"val_eval rows: {len(val_eval):,}")

print("Forecasting test …")
test_eval = recursive_forecast(test_g, test_raw)
print(f"test_eval rows: {len(test_eval):,}")

In [ ]:
def _prices(df):
    return df["dam_price"].values, df["sell_bm_price"].values, df["buy_bm_price"].values

val_smape_v     = smape(val_eval[Y_COL], val_eval['pred'])
val_rmse_v      = rmse(val_eval[Y_COL], val_eval['pred'])
val_mape_v      = mape(val_eval[Y_COL], val_eval['pred'])
val_money_v     = money(val_eval[Y_COL], val_eval['pred'], *_prices(val_eval))
val_money_pct_v = money_pct(val_eval[Y_COL], val_eval['pred'], *_prices(val_eval))

test_smape_v     = smape(test_eval[Y_COL], test_eval['pred'])
test_rmse_v      = rmse(test_eval[Y_COL], test_eval['pred'])
test_mape_v      = mape(test_eval[Y_COL], test_eval['pred'])
test_money_v     = money(test_eval[Y_COL], test_eval['pred'], *_prices(test_eval))
test_money_pct_v = money_pct(test_eval[Y_COL], test_eval['pred'], *_prices(test_eval))

print("── Validation ──────────────────────────────────────────────")
print(f"Aligned samples : {len(val_eval):,}")
print(f"SMAPE     : {val_smape_v:.4f}")
print(f"RMSE      : {val_rmse_v:.4f}")
print(f"MAPE      : {val_mape_v:.2f} %")
print(f"MONEY     : {val_money_v:.4f}")
print(f"MONEY_PCT : {val_money_pct_v:.4f}%")

print("── Test ────────────────────────────────────────────────────")
print(f"Aligned samples : {len(test_eval):,}")
print(f"SMAPE     : {test_smape_v:.4f}")
print(f"RMSE      : {test_rmse_v:.4f}")
print(f"MAPE      : {test_mape_v:.2f} %")
print(f"MONEY     : {test_money_v:.4f}")
print(f"MONEY_PCT : {test_money_pct_v:.4f}%")

mlflow.log_metrics({
    "val_smape":      val_smape_v,
    "val_rmse":       val_rmse_v,
    "val_mape":       val_mape_v,
    "val_money":      val_money_v,
    "val_money_pct":  val_money_pct_v,
    "test_smape":     test_smape_v,
    "test_rmse":      test_rmse_v,
    "test_mape":      test_mape_v,
    "test_money":     test_money_v,
    "test_money_pct": test_money_pct_v,
})
mlflow.end_run()
print(f"MLflow run logged → {mlflow.get_tracking_uri()}")

In [ ]:
def per_station_metrics(eval_df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for grp, gdf in eval_df.groupby(GROUP_COL):
        rows.append({
            GROUP_COL:    grp,
            "n":          len(gdf),
            "SMAPE":      smape(gdf[Y_COL], gdf["pred"]),
            "RMSE":       rmse (gdf[Y_COL], gdf["pred"]),
            "MAPE":       mape (gdf[Y_COL], gdf["pred"]),
            "MONEY":      money(gdf[Y_COL], gdf["pred"], *_prices(gdf)),
            "MONEY_PCT":  money_pct(gdf[Y_COL], gdf["pred"], *_prices(gdf)),
        })
    return pd.DataFrame(rows).sort_values("SMAPE")

test_station_metrics = per_station_metrics(test_eval)

print("Top-10 best stations (test SMAPE):")
print(test_station_metrics.head(10).to_string(index=False))
print("\nBottom-10 worst stations (test SMAPE):")
print(test_station_metrics.tail(10).to_string(index=False))

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

def plot_forecast(df, eic_code, start_dt=None, end_dt=None, title_prefix=""):
    df = df[df[GROUP_COL] == eic_code].sort_values("datetime")
    if df.empty:
        raise ValueError(f"No data for EiC code: {eic_code!r}")
    if start_dt is not None:
        df = df[df["datetime"] >= pd.Timestamp(start_dt)]
    if end_dt is not None:
        df = df[df["datetime"] <= pd.Timestamp(end_dt)]
    if df.empty:
        raise ValueError("No data in the specified datetime range.")

    fig, ax = plt.subplots(figsize=(14, 4))
    ax.plot(df["datetime"], df[Y_COL],  label="True",      linewidth=1, color="steelblue")
    ax.plot(df["datetime"], df["pred"], label="Predicted", linewidth=1, color="tomato", alpha=0.85)
    ax.set_title(
        f"{title_prefix}{eic_code}  |  MAPE={mape(df[Y_COL], df['pred']):.3f}"
        f"  MONEY_PCT={money_pct(df[Y_COL], df['pred'], *_prices(df)):.2f}"
        f"  ({df['datetime'].min().date()} \u2013 {df['datetime'].max().date()})"
    )
    ax.set_xlabel("Datetime")
    ax.set_ylabel(Y_COL)
    ax.legend()
    ax.xaxis.set_major_locator(mdates.AutoDateLocator())
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%m-%d %H:%M"))
    fig.autofmt_xdate(rotation=0, ha="center")
    plt.tight_layout()
    plt.show()

best_station  = test_station_metrics.iloc[0][GROUP_COL]
worst_station = test_station_metrics.iloc[-1][GROUP_COL]

print(f"Best  station (SMAPE): {best_station}")
plot_forecast(test_eval, eic_code=best_station,  title_prefix="[BEST]  ")

print(f"Worst station (SMAPE): {worst_station}")
plot_forecast(test_eval, eic_code=worst_station, title_prefix="[WORST] ")